## 5.	Handle implicit feedback in a recommendation system

In [5]:
pip install implicit

   ---------------------------------------- 0.0/675.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/675.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/675.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/675.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/675.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/675.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/675.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/675.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/675.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/675.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/675.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/675.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/675.3 kB ? eta -:--:--
   ---------------------------------------- 0.0/675.3 kB ? eta -:--:--
   ---

In [6]:
# Import necessary libraries
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
import implicit
import time
from sklearn.model_selection import train_test_split

# --- Step 1: Data Loading and Preprocessing ---
# We'll create a synthetic dataset for demonstration purposes.
# In a real scenario, you would load your data (e.g., from a CSV file)
# which contains user interactions (e.g., user_id, item_id, interaction_strength).

data = {
    'user_id': [1, 1, 1, 2, 2, 3, 3, 3, 3, 4, 4, 4, 5, 5],
    'item_id': [101, 102, 104, 101, 103, 102, 103, 105, 106, 101, 105, 106, 102, 104],
    'interactions': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
}
df = pd.DataFrame(data)

# In implicit feedback, we often have a 'confidence' level instead of a direct rating.
# A higher number of interactions (e.g., clicks, purchases) implies higher confidence in preference.
# For simplicity, we use a constant '1' here to denote an interaction has occurred.

print("Original Data:")
display(df.head())

# --- Step 2: Data Preparation to Sparse Matrix Format ---
# The 'implicit' library requires data in a scipy sparse matrix format (CSR matrix).
# We need to map user and item IDs to unique contiguous integer IDs (0 to N-1).

# Map user and item IDs to continuous indices
user_ids = sorted(df['user_id'].unique())
item_ids = sorted(df['item_id'].unique())
user_map = {user: i for i, user in enumerate(user_ids)}
item_map = {item: i for i, item in enumerate(item_ids)}

df['user_index'] = df['user_id'].map(user_map)
df['item_index'] = df['item_id'].map(item_map)

# Create the sparse user-item interaction matrix
# The matrix will have shape (num_users, num_items)
num_users = len(user_ids)
num_items = len(item_ids)

user_item_matrix = csr_matrix(
    (df['interactions'], (df['user_index'], df['item_index'])),
    shape=(num_users, num_items)
)

print(f"\nSparse User-Item Matrix Shape: {user_item_matrix.shape}")

# --- Step 3: Train-Test Split (Optional but recommended for evaluation) ---
# For implicit feedback, splitting can be tricky. A simple approach is to use
# the 'implicit' library's evaluation tools or manually split before training.
# Here we will just use the full dataset for demonstration of the model fitting.

# --- Step 4: Train the Model using Alternating Least Squares (ALS) ---
# ALS is a common and effective algorithm for implicit feedback datasets.
# The 'implicit' library provides a fast implementation.

# Initialize the ALS model
# factors: Number of latent factors
# regularization: Regularization parameter
# iterations: Number of iterations to run the algorithm
model = implicit.als.AlternatingLeastSquares(
    factors=50,
    regularization=0.01,
    iterations=20,
    num_threads=1 # Adjust based on your system's CPU cores
)

# Train the model on the sparse matrix
print("\nTraining the ALS model...")
start_time = time.time()
model.fit(user_item_matrix)
end_time = time.time()
print(f"Model trained in {end_time - start_time:.2f} seconds.")

# --- Step 5: Generate Recommendations ---

# Example: Recommend items for a specific user (e.g., User ID 1)
target_user_id = 1
target_user_index = user_map[target_user_id]
N = 5 # Number of recommendations to generate

# Generate recommendations
# filter_already_liked_items=True ensures the user's existing interactions aren't recommended again.
recommendations, scores = model.recommend(
    target_user_index,
    user_item_matrix[target_user_index],
    N=N,
    filter_already_liked_items=True
)

# Map item indices back to original item IDs
recommended_item_ids = [item_ids[i] for i in recommendations]

print(f"\nTop {N} recommendations for User {target_user_id}:")
for item_id, score in zip(recommended_item_ids, scores):
    print(f"  Item ID: {item_id}, Score: {score:.4f}")

# --- Step 6: Find similar items (Item-Item Nearest Neighbors) ---
# You can also use the model to find items similar to a given item.

target_item_id = 101
target_item_index = item_map[target_item_id]
N_similar = 3

# Find similar items
similar_items, similar_scores = model.similar_items(target_item_index, N_similar)

# Map item indices back to original item IDs
similar_item_ids = [item_ids[i] for i in similar_items]

print(f"\nTop {N_similar} items similar to Item {target_item_id}:")
for item_id, score in zip(similar_item_ids, similar_scores):
    print(f"  Similar Item ID: {item_id}, Score: {score:.4f}")


Original Data:


,user_id,item_id,interactions
0,1,101,1
1,1,102,1
2,1,104,1
3,2,101,1
4,2,103,1



Sparse User-Item Matrix Shape: (5, 6)

Training the ALS model...


C:\Users\HP-LSC-086\anaconda3\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: Intel MKL BLAS is configured to use 4 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'MKL_NUM_THREADS=1' or by callng 'threadpoolctl.threadpool_limits(1, "blas")'. Having MKL use a threadpool can lead to severe performance issues
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

Model trained in 0.05 seconds.

Top 5 recommendations for User 1:
  Item ID: 103, Score: 0.0028
  Item ID: 106, Score: 0.0014
  Item ID: 105, Score: 0.0014
  Item ID: 104, Score: -340282346638528859811704183484516925440.0000
  Item ID: 102, Score: -340282346638528859811704183484516925440.0000

Top 3 items similar to Item 101:
  Similar Item ID: 101, Score: 1.0000
  Similar Item ID: 104, Score: 0.0621
  Similar Item ID: 103, Score: 0.0351
